""" 
Task 1: Understanding Chat Messages (Conceptual): 
1. What are SystemMessage, HumanMessage, and AIMessage? 
System Message is basically the role we assign to the LLM, HumanMessage is the query given by the User, AI Message is the response received from the LLM
2. Why message-based prompting is better than single prompts?
It is better coz it has history of the chat which helps to model to understand better or retrieve old context from the current query chat
"""

In [5]:
# Task 2- MessagePlaceholder Usage
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()

prompt = ChatPromptTemplate([
    ('system','You are a helpful assistant'),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human','{input}')
])

LLM = ChatGroq(model='openai/gpt-oss-20b')

store = {}

def get_history(session_id: str):
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

chain = prompt | LLM | StrOutputParser()

chatbot = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key='input',
    history_messages_key='chat_history'
)

query = input('Enter The Query')
response_1 = chatbot.invoke(
    {'input':query},
    config = {'configurable':{'session_id':'shrey123'}}
)
response_2 = chatbot.invoke(
    {'input':'What question did i asked?'},
    config = {'configurable':{'session_id':'shrey123'}}
)
print('\n',response_1,'\n',response_2,'\n',store)


d:\miniconda3\envs\genai\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)



 **Elon Musk** (born June 28, 1971) is a South‑African‑born American entrepreneur, inventor, and business magnate best known for founding and leading several high‑profile technology companies. Here’s a quick snapshot of his career and impact:

| Company | Role | Key Contributions |
|---------|------|-------------------|
| **Zip2** (1996) | Co‑founder & CEO | Early online city guide software; sold to Compaq for ~$307 million. |
| **PayPal** (1999) | Co‑founder & CEO | Pioneered online payments; sold to eBay for $1.5 billion. |
| **SpaceX** (2002) | Founder & CEO | Revolutionized space launch with reusable rockets (Falcon 9, Starship). |
| **Tesla, Inc.** (2004) | CEO & Product Architect | Accelerated electric‑vehicle adoption; introduced Model S, 3, X, Y, Cybertruck. |
| **Neuralink** (2016) | Co‑founder | Brain‑computer interface research. |
| **The Boring Company** (2016) | Founder | Tunnel construction & “Hyperloop” concept. |
| **Twitter, Inc.** (2022) | Owner & CEO | Acquired for 

In [15]:
#Task 3: Basic Message History
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langchain_core.prompts import PromptTemplate
messages = [[SystemMessage(content='You are helpful AI Assistent'),
            HumanMessage(content='Who is PM of China?')]]

prompt = PromptTemplate(template='Based on this history continue the conversation {messages} query is {query}',
                        input_variables=['messages','query'])

query = input('Enter the query?')

chain = prompt | LLM

response = chain.invoke({'messages':messages,'query':query})

messages.append([HumanMessage(content=query),AIMessage(content=response.content)])
print(messages)
print(response.content)


[[SystemMessage(content='You are helpful AI Assistent', additional_kwargs={}, response_metadata={}), HumanMessage(content='Who is PM of China?', additional_kwargs={}, response_metadata={})], [HumanMessage(content='I am very rich guy!!', additional_kwargs={}, response_metadata={}), AIMessage(content='That’s great to hear! 💰 How can I help you make the most of it? Are you looking for investment ideas, philanthropic opportunities, business ventures, or something else? Let me know what’s on your mind, and I’ll do my best to provide useful information or suggestions.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]]
That’s great to hear! 💰 How can I help you make the most of it? Are you looking for investment ideas, philanthropic opportunities, business ventures, or something else? Let me know what’s on your mind, and I’ll do my best to provide useful information or suggestions.


In [ ]:
# Task 4: Trimming Chat History
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
messages = [ 
    HumanMessage(content="Hello"), 
    AIMessage(content="Hi there!"), 
    HumanMessage(content="Tell me a story about AI."), 
    AIMessage(content="Once upon a time ... very long response ... "), 
    HumanMessage(content="Can you summarize it?"), 
    AIMessage(content='Sure I can do it so as i was saying that there was a time ... ') 
] 

trimmer = trim_messages(max_tokens=50,token_counter=LLM,include_system=True,allow_partial=False,start_on='human')
prompt = ChatPromptTemplate([
    ('system','You are a helpful assistant'),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human','{input}')
])
chain = (RunnablePassthrough.assign(chat_history=itemgetter('chat_history') | trimmer)
        | prompt | LLM | StrOutputParser())

response = chain.invoke({
    'chat_history':messages,
    'input':input('Enter the Query')
})
print(response)

d:\miniconda3\envs\genai\Lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
d:\miniconda3\envs\genai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\miniconda3\envs\genai\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shrey\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYML

Here are all the assistant’s responses in the order they were given:

---

**Response 1 – Story about AI**

> Once upon a time ... very long response ...  

*(The full story was a whimsical tale about an AI named Liora, a curious child named Milo, and a city that learned to coexist with technology. It covered themes of empathy, curiosity, and the balance between progress and nature. The narrative ended with Milo and Liora looking up at a sky painted with both stars and neon lights, realizing that the future would be shaped by both human hearts and intelligent machines.)*

---

**Response 2 – Summary of the story**

> Sure I can do it so as i was saying that there was a time ...  

*(This answer was a concise recap of the story’s main points, highlighting the characters, setting, key events, and the moral about harmony between humans and AI.)*

---

**Response 3 – Request to list all the responses**

> *No new content – the user asked to list all the responses, so the assistant replied 

In [18]:
trimmer.invoke(messages)

[HumanMessage(content='Tell me a story about AI.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Once upon a time ... very long response ... ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can you summarize it?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sure I can do it so as i was saying that there was a time ... ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

Task 7: Observations & Insights Write short answers: 
1. Why chat history is important ?
To get old conversation for the current answer.
2. Trade-offs between long memory and performance 
Maintaining long conversation history improves contextual understanding but increases token consumption, latency and cost. We must balance memory retention with performance and context-window limits.
3. When to summarize vs trim history 
Trimming removes older messages to fit within a token budget, whereas summarization compresses older conversation history while preserving important information.
4. Difference between message placeholders and memory
message placeholder
MessagesPlaceholder is used to insert conversation history into a prompt, whereas memory or chat history is responsible for storing and retrieving previous conversation messages.
